# Gaze-contingent foveation — the mechanism figure (ch03/ch04, README)

Reproduces `results/foveation_mit1003/gaze_contingent_demo_stim0091.png`.

Two scanpaths are sampled from the pretrained DeepGaze III at the **same seed and
the same start point**, differing only in what the model is allowed to see:

- **normal** — the sharp image;
- **gaze-contingent foveated** — at every step the image is re-foveated around
  the current fixation *before* the forward pass, so the model must choose its
  next saccade from a sharp-at-gaze, low-resolution-but-present peripheral view.

The filmstrip below the paths shows the frames the model actually received. The
point it makes: the fovea tracks the fixation, and the periphery is blurred, not
black. A model that cannot see the periphery at all cannot select a peripheral
target, which would be a different experiment.

> Needs the pretrained DG3 weights (~80 MB, cached by torch hub on first use).
> Forward-only, so MPS is fine — about a minute.

In [ ]:
import runpy
import sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
    REPO = REPO.parent

from IPython.display import Image, display


def run_script(name, *args):
    """Run scripts/<name> exactly as the command line would.

    These notebooks drive the same code the committed figure came from rather
    than reimplementing the plotting, so a notebook cannot silently disagree
    with what is in results/ and in the thesis.
    """
    sys.argv = [name, *[str(a) for a in args]]
    runpy.run_path(str(REPO / 'scripts' / name), run_name='__main__')


def show(*paths, width=1000):
    for p in paths:
        display(Image(filename=str(REPO / p), width=width))

## PARAMETERS — edit these

In [ ]:
STIM_IDX   = 91      # MIT1003 stimulus index; 91 is the running example
SEED       = 0       # same seed for both arms — the paths diverge because the input differs
N_FIX      = 10      # fixations to sample per arm
FRAMES     = 6       # foveated frames in the filmstrip
PPD        = 35.0    # MIT1003 viewing geometry (pixels per degree)
FOVEAL_CPD = 20.0    # committed figure uses 20 (stronger than human, so the blur is legible)

`FOVEAL_CPD = 20` is a presentation choice, not the primary arm. At the human
value of 40 the sharp disc is 2.96° wide and the peripheral blur is hard to see
in a printed figure. The measured primary contrast is cpd 40; see ch04.

In [ ]:
run_script('demo_foveated_scanpath.py',
           '--stim-idx', STIM_IDX,
           '--seed', SEED,
           '--n-fix', N_FIX,
           '--frames', FRAMES,
           '--ppd', PPD,
           '--foveal-cpd', FOVEAL_CPD)

In [ ]:
show(f'results/foveation_mit1003/gaze_contingent_demo_stim{STIM_IDX:04d}.png', width=1200)

## The sidecar

Every figure in this repo writes a JSON sidecar next to it recording the seed,
the stimulus, the start point, the device and the foveation record — so a figure
can always be traced back to the run that made it. Sampling is stochastic: a
rerun on a different device may draw a different path from the same
distribution, which is why the sidecar records both the seed and the resulting
coordinates.

In [ ]:
import json
p = REPO / f'results/foveation_mit1003/gaze_contingent_demo_stim{STIM_IDX:04d}.json'
print(json.dumps(json.loads(p.read_text()), indent=2)[:1600])

## Command line

```bash
.venv/bin/python scripts/demo_foveated_scanpath.py --stim-idx 91
.venv/bin/python scripts/demo_foveated_scanpath.py --stim-idx 91 --foveal-cpd 12
```

---

# Where does foveation move the target, and where does it not?

Reproduces `results/foveation_mit1003/next_pixel_check_stim0091.png`.

The demo above shows *that* the two arms diverge. This asks *when*. Najemnik &
Geisler (2005) predict the answer: saccade targets are selected from peripheral
preview, so degrading the periphery should matter only when the thing worth
looking at is **in** the periphery. If the fovea already sits on the scene's
attractor, foveation removes nothing the model was using and the predicted target
should not move at all.

`next_pixel_check.py` tests that instead of illustrating it. It sweeps gaze over a
grid, takes the mode of the log-density under sharp and foveated input at every
point, and regresses the mode shift on the distance from gaze to the attractor.
The two illustrative panels are then picked *from the sweep* — the gaze point
nearest the attractor, and the one with the largest measured shift — so the
picture cannot disagree with the measurement.

> ~70 forward passes; a few minutes on MPS. The blur pyramid is built once and
> shared across all gaze points.

In [ ]:
GRID       = (7, 5)    # gaze sweep resolution (nx, ny)
CHECK_CPD  = 20.0      # foveation strength for the check
CHUNK      = 8         # gaze points per forward batch — lower this if memory is tight

run_script('next_pixel_check.py',
           '--stim-idx', STIM_IDX,
           '--grid', GRID[0], GRID[1],
           '--ppd', PPD,
           '--foveal-cpd', CHECK_CPD,
           '--chunk', CHUNK)

In [ ]:
show(f'results/foveation_mit1003/next_pixel_check_stim{STIM_IDX:04d}.png', width=1150)

The sidecar carries the whole sweep, not just the two panels — so the claim can be
re-checked without rerunning the model.

In [ ]:
d = json.loads((REPO / f'results/foveation_mit1003/next_pixel_check_stim{STIM_IDX:04d}.json').read_text())
print(f"attractor            : {d['attractor_xy']}")
print(f"Spearman rho         : {d['spearman_rho']:.3f}  (p = {d['spearman_p']:.2g})")
print(f"moved, gaze NEAR attr: {d['n_moved_near_attractor']}/{d['n_near_attractor']}")
print(f"moved, gaze FAR  attr: {d['n_moved_far_attractor']}/{d['n_far_attractor']}")
print(f"median / max shift   : {d['median_shift_px']:.0f} / {d['max_shift_px']:.0f} px")

Read the median alongside the maximum. Most gaze positions show **no** shift —
that is the point, not a weakness: the effect is confined to the positions the
theory says it should be confined to. Quoting only the extreme (an earlier version
of this figure reported "~6 px vs ~426 px") makes a conditional effect look like a
typical one.